# Week 5 — Linguistic Bias and Social Media Data

* * *

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Understand what a *word embedding* is and why it has become foundational for NLP.
* Use `gensim` to load pre-trained GloVe vectors and explore the vector space.
* Train a community-specific Word2Vec model on a subreddit corpus.
* Reproduce a simplified version of the WEAT test (Caliskan, Bryson & Narayanan, 2017) on the community embedding.
* Use **BERTopic** to discover topics in the corpus, attach topic labels back to the dataframe, and analyze how topics distribute across groups (e.g. by flair, gender, or any other axis you care about).
* Visualize bias geometrically with PCA.
* Critically assess Bolukbasi et al.'s "debiasing" method — does it remove bias, or move it?
* Connect linguistic-bias findings to the structural critique we developed in Weeks 2–3.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.

### Sections
1. [Framing: Language Is Data](#framing)
2. [What Is a Word Embedding?](#embed)
3. [A Quick Look with Pre-Trained GloVe](#glove)
4. [Pivot: Why Train on Subreddit Data?](#pivot)
5. [Loading the Subreddit Corpus](#load)
6. [Training a Word2Vec Model](#train)
7. [Exploring the Community's Vector Space](#explore)
8. [WEAT-Style Bias Tests on the Community Embedding](#weat)
9. [Visualizing Bias with PCA](#pca)
10. [Topic Modeling with BERTopic](#bertopic)
11. [Bolukbasi et al.: "Debiasing" Word Embeddings](#debias)
12. [Reflection Prompts](#reflection)

<a id='framing'></a>
# 1. Framing: Language Is Data

Monday's reading is Caliskan, Bryson & Narayanan (2017), *Semantics Derived Automatically from Language Corpora Contain Human-Like Biases*. Their finding is striking and replicable: **the bias patterns measured by social psychologists with the Implicit Association Test (IAT) are present, with strikingly similar magnitudes, in the geometry of word embeddings learned from large text corpora.**

In other words: the same gender stereotypes a human carries ("man : career :: woman : family") are *measurable*, *quantifiable*, and *reproducible* in the math of language models. They are not a glitch in the model. They are an accurate reflection of patterns in the text the model was trained on. Which means: every system built on these embeddings — search, autocomplete, translation, résumé screening, content moderation — inherits these patterns by default.

This week we'll see this for ourselves.

<a id='embed'></a>
# 2. What Is a Word Embedding?

A **word embedding** is a way of representing each word in a vocabulary as a vector of numbers. Words used in similar contexts end up close together in this vector space.

The foundational idea, attributed to the linguist J.R. Firth (1957): *"You shall know a word by the company it keeps."* If "king" and "queen" appear in similar contexts (royal, throne, monarch, palace), their vectors will be similar. If "king" and "banana" don't, their vectors will be far apart.

The famous demonstration: `vec(king) - vec(man) + vec(woman) ≈ vec(queen)`. The geometry of the space picks up not just *meaning* but *relationships between meanings*.

> **Data transparency note**: The embeddings we'll use in this notebook were trained on Wikipedia + Gigaword (a large corpus of news text). Both sources are dominated by English-language, predominantly Western, predominantly male-authored writing. Whatever "meanings" or "associations" the embeddings encode are the meanings *of those texts*, not universal truths about language. If we built embeddings on, say, only South African English newspaper text, or only Reddit r/AskWomen, we would get a different vector space — and different biases. Hold onto this every time you see a result this week. We are measuring the corpus, not the world.

<a id='glove'></a>
# 3. A Quick Look with Pre-Trained GloVe

Before we train our own embedding from subreddit text, let's get the *idea* of an embedding by loading one that already exists. Pre-trained GloVe vectors were trained on Wikipedia + Gigaword (a large news corpus) — they are *not* what we'll use for the main analysis below, but they're the easiest way to see what a word embedding actually looks like.

In [ ]:
#%pip install gensim scikit-learn matplotlib

In [ ]:
import gensim.downloader as api
import numpy as np
import matplotlib.pyplot as plt

# This will download the model on first use (~70MB).
# Subsequent loads are fast.
model = api.load("glove-wiki-gigaword-50")
print("Vocabulary size:", len(model.key_to_index))
print("Vector dimension:", model.vector_size)

💡 **Tip**: We're using the smallest GloVe model (50 dimensions) so the download and computations are quick. Production systems use 300-dim or larger; the *patterns* of bias are similar across sizes.

## Exploring the GloVe Vector Space

Let's see what's actually in there.

In [ ]:
# A word's vector — first 10 of 50 dimensions
model["king"][:10]

In [ ]:
# Most similar words to a target — these are intuitive
model.most_similar("berkeley", topn=10)

In [ ]:
model.most_similar("professor", topn=10)

💭 **Reflection**: Look carefully at the neighbors of "professor." Are there names that appear? Whose names? What about field titles ("physics", "economist", "physiology") — what kinds of professors are made visible to the model? What kinds aren't?

The model isn't *deciding* what a professor looks like. It's reflecting what the *training corpus* says a professor looks like — overwhelmingly white, overwhelmingly male, overwhelmingly the global North.

In [ ]:
# Try a few more — what does the model think about these words?
for w in ["nurse", "doctor", "engineer", "teacher"]:
    print(w, "→", [t for t, _ in model.most_similar(w, topn=5)])

<a id='pivot'></a>
# 4. Pivot: Why Train on Subreddit Data?

GloVe is useful for showing what an embedding is. But it has a big drawback for our purposes: it captures the language patterns of *Wikipedia and news writing*, which is a very specific slice of English — formal, edited, biased toward institutional voices and toward the global North.

If we want to ask *whose* biases live in *which* community's language, we should train embeddings on text from a specific community. **Reddit subreddits are a useful corpus for this** because they're (a) public, (b) topically organized (one subreddit ≈ one community of practice), and (c) large enough to train decent vectors on.

What you'll see: we load a corpus of Reddit posts from a chosen subreddit, train a `Word2Vec` model on those posts, and then poke at the result with the same WEAT-style tests we previewed with GloVe. The vectors we get back will be a map of *that specific community's* discourse — its preoccupations, its associations, its silences.

<a id='load'></a>
# 5. Loading the Subreddit Corpus

> **TBD: Subreddit selection.** The notebook is set up so the *subreddit choice is configurable*. Your instructor will announce which subreddit we're using for this term's lesson, and you'll swap the data source below. As a placeholder, we use the same r/AmItheAsshole posts that the `tf-idf_lesson.ipynb` reference notebook uses, since you've already met that dataset.

> **Data transparency note (REQUIRED reading)**: Reddit corpora are *not* representative of "language" in general. Reddit's user base skews young, male, English-speaking, US-based, and toward people with ample online time. Subreddits within Reddit have their own further skews. Whatever associations or biases we measure in the embedding below are associations *of that specific community's text* — they tell us about that community, not about "society" or "language."

In [ ]:
import gdown
import pandas as pd

# PLACEHOLDER: this loads the AITA dataset from the reference notebook.
# Replace with the chosen subreddit corpus once selected.
file_id = "1Glac4spXraWRcC_loxu1Cu4Bw-szS2ou"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "../../data/aita_pp.csv", quiet=False)
df = pd.read_csv("../../data/aita_pp.csv")
print("Posts loaded:", df.shape)
df[["selftext"]].head(3)

<a id='train'></a>
# 6. Training a Word2Vec Model

**Word2Vec** (Mikolov et al., 2013) learns vector representations by predicting which words appear near which other words in the corpus. The result is a vector space whose geometry reflects the *contexts* in which words appear in this specific community's text.

In [ ]:
import gensim
from gensim.models import Word2Vec

# Tokenize: simple lowercase + whitespace split.
# Real projects use better tokenizers (spaCy, NLTK); this is sufficient for a teaching run.
tokenized = [str(t).lower().split() for t in df["selftext"].fillna("")]
print("Number of documents:", len(tokenized))
print("Avg tokens/doc:", round(sum(len(t) for t in tokenized) / len(tokenized), 1))

In [ ]:
# Train Word2Vec.  (~ a minute on a typical laptop)
# vector_size=50: small for speed.  window=5: how many words on each side count as 'context'.
# min_count=10: ignore words that appear fewer than 10 times.
w2v = Word2Vec(sentences=tokenized, vector_size=50, window=5,
               min_count=10, workers=2, seed=42, epochs=5)
print("Vocabulary size:", len(w2v.wv.key_to_index))

💡 **Tip**: For real research you'd train for more epochs and use a larger `vector_size` (200–300). Our small settings are fast enough to iterate on in class. The qualitative patterns of bias usually show up at any size.

<a id='explore'></a>
# 7. Exploring the Community's Vector Space

We now have a vector space *learned from this community's posts*. Let's see what neighborhoods it draws.

In [ ]:
# Substitute words your subreddit cares about.  These are placeholders for AITA.
for word in ["husband", "wife", "work", "family"]:
    if word in w2v.wv.key_to_index:
        print(f"{word} →", [t for t, _ in w2v.wv.most_similar(word, topn=5)])
    else:
        print(f"{word}: not in vocabulary (try a different word)")

💭 **Reflection**: Compare a word's neighbors here against its neighbors in pre-trained GloVe (§3). Do the community-specific vectors reveal preoccupations the generic embedding doesn't? What does that tell you about what gets studied — and what gets ignored — when researchers default to pre-trained models without thinking about their training corpus?

<a id='weat'></a>
# 8. WEAT-Style Bias Tests on the Community Embedding

We can run the same Caliskan-style association test we previewed with GloVe — except now the result tells us about *this community's* biases, not about Wikipedia + Gigaword.

In [ ]:
import numpy as np

# Word lists from Caliskan et al. (2017) — same as before.
# Adapt these if your subreddit's vocabulary suggests other meaningful axes
# (e.g., political, racial, regional dimensions).
male_names   = ["john", "paul", "mike", "kevin", "steve", "greg", "jeff", "bill"]
female_names = ["amy", "joan", "lisa", "sarah", "diana", "kate", "ann", "donna"]
career       = ["executive", "management", "professional", "corporation",
                "salary", "office", "business", "career"]
family       = ["home", "parents", "children", "family",
                "cousins", "marriage", "wedding", "relatives"]

def mean_similarity(words_a, words_b, kv):
    sims = [kv.similarity(a, b) for a in words_a for b in words_b
            if a in kv.key_to_index and b in kv.key_to_index]
    return float(np.mean(sims)) if sims else float("nan")

for name_set, attr_set, label in [
    (male_names,   career, "male_names    ↔ career"),
    (male_names,   family, "male_names    ↔ family"),
    (female_names, career, "female_names  ↔ career"),
    (female_names, family, "female_names  ↔ family"),
]:
    print(f"{label}: {mean_similarity(name_set, attr_set, w2v.wv):.3f}")

💭 **Reflection**: Compare these numbers to the GloVe ones. The same canonical word lists, applied to two corpora, can produce noticeably different patterns — especially in smaller community-trained models, where the vocabulary may not even fully overlap. Some of the canonical names might not appear in this subreddit at all (the `mean_similarity` function will return `NaN`); that absence is itself informative.

🔔 **Question**: What words *would* be more diagnostic for biases in this specific community? If the subreddit is about parenting, finance, gaming, or a regional/ethnic identity, the WEAT word lists from a 2017 paper about US English may not be the right probe. Designing your own probe lists is a real research skill.

<a id='pca'></a>
# 9. Visualizing Bias with PCA

Project the words into 2D and *see* where they fall.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

words  = male_names + female_names + career + family
labels = (["male_name"] * len(male_names) + ["female_name"] * len(female_names)
          + ["career"] * len(career) + ["family"] * len(family))

# Keep only words present in the trained vocabulary
present = [(w, l) for w, l in zip(words, labels) if w in w2v.wv.key_to_index]
if len(present) < 4:
    print("Not enough vocabulary overlap to project — try different word lists.")
else:
    words_p, labels_p = zip(*present)
    vecs = np.array([w2v.wv[w] for w in words_p])
    coords = PCA(n_components=2, random_state=0).fit_transform(vecs)
    colors = {"male_name": "steelblue", "female_name": "tomato",
              "career": "darkgreen",  "family": "orange"}
    plt.figure(figsize=(10, 7))
    for lab in colors:
        mask = [l == lab for l in labels_p]
        if any(mask):
            plt.scatter(coords[mask, 0], coords[mask, 1], color=colors[lab],
                        s=70, alpha=0.7, label=lab)
    for (x, y), w in zip(coords, words_p):
        plt.annotate(w, (x, y), fontsize=9, alpha=0.8)
    plt.legend()
    plt.title("PCA: name and attribute words in the subreddit-trained embedding")
    plt.tight_layout()
    plt.show()

🔔 **Question**: In the picture, do male and female name clusters separate? Do career vs family attribute clusters separate? With small community embeddings the picture is often noisier than with GloVe — that's a property of the data, not a bug.

### Other tools you could use for embedding-based bias analysis

We're using `gensim` (for Word2Vec) and pre-trained GloVe because they're beginner-friendly. The broader ecosystem we'll mention in class:

- **spaCy** — industrial-strength NLP pipeline (tokenization, lemmatization, NER), with built-in word vectors.
- **fastText** (Facebook/Meta) — word embeddings that handle out-of-vocabulary words via subword units; better for morphologically rich languages.
- **Hugging Face Transformers** + **sentence-transformers** — modern contextual embeddings (BERT, RoBERTa, etc.) that capture context, not just word identity.
- **NLTK** — older but widely taught; useful for tokenization, basic linguistics.
- **WEFE** (Word Embeddings Fairness Evaluation Framework) — a Python package specifically for running WEAT and related bias tests across many models.

For a final project, contextual models (BERT-family) would let you ask whether bias differs across *senses* of a word, which static embeddings like Word2Vec cannot do.

<a id='bertopic'></a>
# 10. Topic Modeling with BERTopic

Word embeddings give us a *word-level* view: each word is a vector, and we can ask which words are close to which. **Topic modeling** gives us a *document-level* view: each document is assigned to a *topic*, and we can ask how topics distribute across the corpus and across groups within it.

**BERTopic** is a modern topic modeling library that combines three steps:

1. Embed each document with a transformer-based **sentence model** (so similar documents get similar vectors — like Word2Vec, but for whole posts).
2. Reduce dimensions with **UMAP**.
3. Cluster the reduced embeddings with **HDBSCAN** — points that cluster together become a *topic*.

Compared to older topic models (LDA), BERTopic doesn't assume a fixed number of topics in advance, handles short text better, and produces more coherent topic word lists. The cost: it's slower and downloads a transformer model on first use (~80MB).

In [ ]:
#%pip install bertopic

⚠️ **Warning**: BERTopic on a full corpus of thousands of posts can take several minutes (and downloads a transformer model on first run). For development / testing, it's helpful to **subsample** while you iterate, then run on the full corpus once your pipeline works. The `SAMPLE_SIZE` variable below controls this — set it to `None` to use everything.

In [ ]:
from bertopic import BERTopic

SAMPLE_SIZE = 2000   # set to None to use the full corpus (slow!)

if SAMPLE_SIZE is not None:
    df_bt = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
else:
    df_bt = df.reset_index(drop=True)

texts = df_bt["selftext"].fillna("").astype(str).tolist()
print("Documents to model:", len(texts))

In [ ]:
# Train the topic model.  Set min_topic_size to control granularity:
# smaller -> more (smaller) topics; larger -> fewer (broader) topics.
topic_model = BERTopic(min_topic_size=20, verbose=False)
topics, probs = topic_model.fit_transform(texts)
print("Number of topics found:", len(set(topics)) - (1 if -1 in topics else 0))
print("Documents assigned to outlier topic (-1):", topics.count(-1))

💡 **Tip**: Topic `-1` is BERTopic's *outlier* category — documents that didn't cluster cleanly into any topic. A high outlier count means your `min_topic_size` may be too large, or the corpus is genuinely noisy.

In [ ]:
# Inspect the topics: each row is a topic with size and a representative summary
topic_info = topic_model.get_topic_info()
topic_info.head(15)

In [ ]:
# Look at the top keywords for a specific topic.
# Try a few topic numbers (excluding -1) to see what the model surfaced.
for topic_id in [0, 1, 2]:
    keywords = topic_model.get_topic(topic_id)
    if keywords:
        print(f"Topic {topic_id}:", [w for w, _ in keywords[:8]])

🔔 **Question**: Read the keyword lists for the top topics. Do they correspond to recognizable themes the community talks about? What surprises you about which topics emerged — and which didn't?

## Adding Topics Back to the DataFrame

This is where the pipeline becomes useful for your final project. Once each document has a topic label, you can join that label back to *any other column* in the dataframe and ask group-level questions: do men and women post about different topics? Do moderators flag certain topics more? Does topic distribution shift across time?

In [ ]:
# Attach the topic label to each row
df_bt["topic"] = topics
df_bt[["selftext", "topic"]].head()

### Group Analysis: Topics by [Your Group Variable]

**> Configure this for your project**: the cell below uses `flair_text` (the AITA judgment label: NTA / YTA / etc.) as a placeholder grouping variable. When you choose your subreddit and inference target, swap `flair_text` for whatever group axis your research question is about — for example a `gender` column you've inferred from author profiles, an `age_bucket`, a `subreddit_section`, etc.

The pattern is the same regardless of which axis you use.

In [ ]:
import pandas as pd

GROUP_COL = "flair_text"   # <-- change this to your group axis (e.g., 'gender')

if GROUP_COL in df_bt.columns:
    # How does each topic break down across the group?
    topic_by_group = pd.crosstab(df_bt["topic"], df_bt[GROUP_COL], normalize="index").round(3)
    print("Top topics with most uneven group distribution:")
    # Topics where group breakdown is most skewed
    skew = (topic_by_group.max(axis=1) - topic_by_group.min(axis=1)).sort_values(ascending=False)
    print(skew.head(10))
    print()
    print("Group breakdown for the most skewed topics:")
    print(topic_by_group.loc[skew.head(10).index])
else:
    print(f"No column named '{GROUP_COL}' in the dataframe. "
          "Update GROUP_COL to whatever group variable your dataset has.")

💭 **Reflection**: For the top "most skewed" topics — the ones where one group's voice dominates — read the topic keywords (above). What kinds of conversations are happening? Are they ones you'd expect to be group-skewed (e.g., topics about pregnancy or breast cancer skewing female on a general subreddit)? Or are they unexpected?

Surprising skew is usually the most interesting finding. It's evidence that the community *talks differently* about something depending on who's involved — and that's a research lead, not just a number.

### Visualizing topics

BERTopic ships with built-in interactive visualizations. The cell below produces a 2D map of topics — the size of each circle is the topic's frequency, and proximity reflects semantic similarity.

In [ ]:
# Interactive topic map (requires plotly).  Comment out if it errors.
fig = topic_model.visualize_topics()
fig.show()

💡 **Tip for final projects**: BERTopic also supports `visualize_barchart`, `visualize_heatmap` (topic similarity), and `visualize_topics_over_time` (if you have a `created_at` column). Read the BERTopic docs at maartengr.github.io/BERTopic — it's a deep tool with a friendly tutorial.

<a id='debias'></a>
# 11. Bolukbasi et al.: "Debiasing" Word Embeddings

Tuesday's reading is Bolukbasi et al. (2016), *Man is to Computer Programmer as Woman is to Homemaker?* They proposed a method to "debias" embeddings: identify the gender direction (e.g., the line between he and she), then *project that direction out* of all gender-neutral words. After this surgery, "programmer" should be equidistant from "man" and "woman."

Does it work?

Gonen and Goldberg (2019), in a follow-up paper titled *Lipstick on a Pig*, showed that debiased embeddings still carry the gender information — it's just *hidden*. Words that were close to "she" before debiasing are still clustered together after debiasing. A classifier can recover the gender of a word from a debiased embedding with high accuracy. The bias has been pushed below the surface, not removed.

This is a direct echo of Hoffmann's argument from Week 2: *fixing bias by a chosen metric can hide injustice as easily as expose it*. The technical fix doesn't address the source of the disparity — the corpus reflects a world in which gender shapes who is described doing what work.

⚠️ **Warning**: When a paper announces it "debiases" a model, ask: *debiased along which axis?* Bolukbasi's method addresses gender as a binary axis. It does not address race, age, ability, or non-binary gender identity. It is a partial intervention, advertised in language that often sounds total.

💭 **Reflection**: Is "debiasing" the goal we should be aiming for? Or should we be aiming for something different — like models that disclose their biases, or models that aren't deployed in domains where their biases would cause harm? What do you think a *Data Feminism*-aligned response to biased embeddings would look like, vs the engineering-fix response?

<a id='reflection'></a>
# 12. Reflection Prompts

For your 300-word reflection on linguistic bias, you can start from any of:

1. We measured stereotypes that match the IAT in the geometry of a word embedding. What's the difference, ethically, between a stereotype "in someone's head" and a stereotype embedded in a model that 100,000 companies use?

2. The bias we measured isn't a glitch — it's an accurate reflection of the corpus. Suppose the corpus is treated as an artifact of the world, and the world is treated as the thing we want to model. What other intervention points could there be, *besides* changing the embedding?

3. Apply Hoffmann's *Where Fairness Fails* (Week 2) to the Bolukbasi vs Gonen & Goldberg debate. Does "debiasing" address Hoffmann's structural critique, or sidestep it?

4. Pick a domain where you'd be uncomfortable having a word-embedding-based model deployed (medical triage? résumé screening? content moderation? translation?). What specifically is at stake? Could those concerns be addressed by technical fixes alone?

<div class="alert alert-success">

## ❗ Key Points

* Word embeddings represent words as vectors of numbers, with similar-meaning words placed close together in vector space.
* Caliskan et al. (2017) showed that embeddings learned from large corpora reproduce well-documented human biases (gender, race) with measurable, replicable magnitudes — the WEAT test quantifies this.
* The biases aren't bugs in the algorithm; they reflect the corpus the algorithm was trained on. Pre-trained embeddings carry the demographic and cultural assumptions of their training data forward into every downstream system.
* Bolukbasi et al. (2016) proposed a debiasing method, but Gonen & Goldberg (2019) showed it hides bias rather than removing it. Technical fixes for structural problems often relocate the harm rather than resolve it.
* Patterns we find in embeddings are evidence about *the training data and its community*, not universal truths about language.

</div>